In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load the Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

# 2. Define Target, Leaks, and ID Column
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'

# These are the features causing the 0.9+ R2 data leakage. 
# We MUST drop them from both the training and testing sets.
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

# Prepare Training Data
y_train = train_df[TARGET]
X_train = train_df.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

# Prepare Testing Data (Save IDs for the final submission format)
test_ids = test_df[ID_COL]
X_test = test_df.drop(columns=[ID_COL] + LEAKED_FEATURES)

# 3. Build a Preprocessing Pipeline
# Identify which columns are text/categorical and which are numbers
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# Fill missing numbers with the median
numeric_transformer = SimpleImputer(strategy='median')

# Fill missing text with the most frequent value, then convert to numbers.
# handle_unknown='use_encoded_value' ensures the model doesn't crash if 
# the test set contains a category it never saw during training.
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 4. Define and Train the Model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

print("Training model (this might take a moment)...")
model.fit(X_train, y_train)

# 5. Predict on Test Data and Format Submission
print("Predicting on test data...")
predictions = model.predict(X_test)

# Create the final dataframe matching the sample_submission format
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: predictions
})

# Save to CSV
submission.to_csv('submissionDay9.csv', index=False)
print("Saved predictions to 'submission.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_31412\3626102385.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training model (this might take a moment)...
Predicting on test data...
Saved predictions to 'submission.csv'


In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Build the Feature Engineering Function
def engineer_features(df):
    data = df.copy()
    
    # Feature 1: Speed Advantage (Who moves first)
    data['speed_advantage'] = data['speed_stat_pikachu'] - data['speed_stat_opponent']
    
    # Feature 2: Level Ratio (Relative Power)
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5) # Added a tiny number to prevent divide by zero
    
    # Feature 3: Attack/Defense Matchup Ratio
    # If the move is Special, compare sp_attack to sp_defense. 
    # Otherwise, compare regular attack to defense.
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['attack_stat'] / (data['defense_stat'] + 1e-5)
    )
    
    # Feature 4: Theoretical Power of the Turn
    # Combine move power, effectiveness, and stat ratios into one number
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    # Feature 5: Previous HP State
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)

    danger = np.ones(len(data))
    
    # If the weather matches the opponent's type, their attacks will hit Pikachu much harder
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    
    # Hail and Sandstorm deal chip damage every turn to Pikachu, lowering expected HP
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    
    # If Electric Terrain is active, Pikachu gets a power boost, lowering the danger
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    
    data['weather_danger_level'] = danger
        
    return data

# Apply the new features to both datasets
print("Engineering features...")
train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target and Remove Leaks
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)

# 4. Build Preprocessing Pipeline
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 5. Define Model and Train
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

print("Training model with new features...")
model.fit(X_train, y_train)

# 6. Predict and Create Submission
print("Predicting on test data...")
predictions = model.predict(X_test)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: predictions
})

submission.to_csv('submissionDay9.csv', index=False)
print("Saved predictions to 'engineered_submission.csv'")

Engineering features...
Training model with new features...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_31412\1263305439.py:77: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Predicting on test data...
Saved predictions to 'engineered_submission.csv'


In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Feature Engineering Function
def engineer_features(df):
    data = df.copy()
    
    # 2.1 Stat Differentials
    data['speed_advantage'] = data['speed_stat_pikachu'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    # 2.2 Attack/Defense Ratios based on Physical vs Special moves
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['attack_stat'] / (data['defense_stat'] + 1e-5)
    )
    
    # 2.3 Theoretical Power combination
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    # 2.4 Relative HP state
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
    # 2.5 Weather Danger Level Mapping
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger


    data['adj_speed_pikachu'] = np.where(data['pikachu_status'] == 'Paralyzed', 
                                         data['speed_stat_pikachu'] * 0.5, 
                                         data['speed_stat_pikachu'])
    
    data['adj_attack_pikachu'] = np.where(data['pikachu_status'] == 'Burned', 
                                          data['attack_stat'] * 0.5, 
                                          data['attack_stat'])

    # 1. Stat Differentials (Using Adjusted Speed)
    data['speed_advantage'] = data['adj_speed_pikachu'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    # 2. Attack/Defense Ratios (Using Adjusted Attack)
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack_pikachu'] / (data['defense_stat'] + 1e-5)
    )
    
    # --- NEW: STAB (Same Type Attack Bonus) ---
    # Pikachu is Electric. If the move is Electric, it hits 1.5x harder.
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    
    # 3. Theoretical Power (Now including STAB)
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['stab_multiplier'] * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    # 4. Relative HP state
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
    # 5. Weather Danger Level Mapping
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger

    return data

# Apply feature engineering
print("Engineering features...")
train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target and Remove Leaks
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)

# FILL MISSING MAX_HP to prevent NaNs during the clipping step
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Build Preprocessing Pipeline
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 5. Define Gradient Boosting Model
hgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(
        max_iter=300, 
        learning_rate=0.05, 
        max_depth=6, 
        random_state=42
    ))
])

# Train
print("Training Gradient Boosting model...")
hgb_model.fit(X_train, y_train)

# 6. Predict and Post-Process (Clipping)
print("Predicting and applying physics constraints...")
raw_predictions = hgb_model.predict(X_test)

# Force predictions to be physically possible (0 to max_hp)
clipped_predictions = np.clip(raw_predictions, 0, test_max_hp)

# Create submission
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})

submission.to_csv('submissionDay10.csv', index=False)
print("Success! Saved clean predictions to 'SubmissionDay10.csv'")

Engineering features...
Training Gradient Boosting model...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_25748\1336641661.py:113: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Predicting and applying physics constraints...
Success! Saved clean predictions to 'SubmissionDay10.csv'


In [ ]:
# import pandas as pd  used to combination of random trees and xgboost to avg out the errors and find out the correct output but did not work that well
# import numpy as np   so now increased the weight of xgboost but that backfired as well so xgboost is not the correct model iguess so switched to the 
# import xgboost as xgb  og histboost 
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OrdinalEncoder

# # 1. Load Data
# train_df = pd.read_csv('train.csv')
# test_df = pd.read_csv('test.csv')

# # 2. Feature Engineering
# def engineer_features(df):
#     data = df.copy()
#     data['speed_advantage'] = data['speed_stat_pikachu'] - data['speed_stat_opponent']
#     data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
#     data['attack_defense_ratio'] = np.where(
#         data['move_category'] == 'Special',
#         data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
#         data['attack_stat'] / (data['defense_stat'] + 1e-5)
#     )
    
#     data['theoretical_power'] = (
#         data['move_power'].fillna(0) * 
#         data['type_effectiveness'].fillna(1.0) * 
#         data['attack_defense_ratio'].fillna(1.0)
#     )
    
#     if 'previous_hp' in data.columns and 'max_hp' in data.columns:
#         data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
#     danger = np.ones(len(data))
#     danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
#     danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
#     danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
#     danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
#     data['weather_danger_level'] = danger
    
#     return data

# print("Engineering features...")
# train_feat = engineer_features(train_df)
# test_feat = engineer_features(test_df)

# # 3. Setup Target, Remove Leaks, and Fill Missing max_hp
# TARGET = 'pikachu_hp'
# ID_COL = 'battle_turn'
# LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

# y_train = train_feat[TARGET]
# X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

# test_ids = test_feat[ID_COL]
# X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)
# test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# # 4. Build Preprocessing Pipeline
# categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
# numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# numeric_transformer = SimpleImputer(strategy='median')
# categorical_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='most_frequent')),
#     ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
# ])

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', numeric_transformer, numeric_cols),
#         ('cat', categorical_transformer, categorical_cols)
#     ])

# # 5. Define BOTH Models
# print("Initializing models...")
# rf_model = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', RandomForestRegressor(
#         n_estimators=150, 
#         max_depth=12, 
#         random_state=42, 
#         n_jobs=-1
#     ))
# ])

# xgb_model = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', xgb.XGBRegressor(
#         n_estimators=300, 
#         learning_rate=0.05, 
#         max_depth=6, 
#         subsample=0.8,
#         colsample_bytree=0.8,
#         random_state=42,
#         n_jobs=-1
#     ))
# ])

# # 6. Train Models
# print("Training Random Forest...")
# rf_model.fit(X_train, y_train)

# print("Training XGBoost...")
# xgb_model.fit(X_train, y_train)

# # 7. Predict and Average (The Ensemble)
# print("Generating predictions...")
# rf_preds = rf_model.predict(X_test)
# xgb_preds = xgb_model.predict(X_test)

# # Give XGBoost 85% weight, and Random Forest 15% weight
# ensemble_preds = (0.85 * xgb_preds) + (0.15 * rf_preds)

# # Post-processing: Force physical boundaries
# clipped_preds = np.clip(ensemble_preds, 0, test_max_hp)

# # 8. Create Submission
# submission = pd.DataFrame({
#     ID_COL: test_ids,
#     TARGET: clipped_preds
# })

# submission.to_csv('submissionDay10.csv', index=False)
# print("Success! Saved clean predictions to 'submissionDay10.csv'")

Engineering features...
Initializing models...
Training Random Forest...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_25748\2312905063.py:61: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training XGBoost...
Generating predictions...
Success! Saved clean predictions to 'ensemble_xgb_rf_submission.csv'


In [ ]:
# import pandas as pd  adopted a new ouput mechanism instead of calculating the final hp we are finding the damage dealt and it backfired
# import numpy as np 
# from sklearn.ensemble import HistGradientBoostingRegressor
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OrdinalEncoder

# # 1. Load Data
# train_df = pd.read_csv('train.csv')
# test_df = pd.read_csv('test.csv')

# # 2. Feature Engineering (Keeping all your powerful mechanics!)
# def engineer_features(df):
#     data = df.copy()
    
#     data['adj_speed_pikachu'] = np.where(data['pikachu_status'] == 'Paralyzed', data['speed_stat_pikachu'] * 0.5, data['speed_stat_pikachu'])
#     data['adj_attack_pikachu'] = np.where(data['pikachu_status'] == 'Burned', data['attack_stat'] * 0.5, data['attack_stat'])

#     data['speed_advantage'] = data['adj_speed_pikachu'] - data['speed_stat_opponent']
#     data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
#     data['attack_defense_ratio'] = np.where(
#         data['move_category'] == 'Special',
#         data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
#         data['adj_attack_pikachu'] / (data['defense_stat'] + 1e-5)
#     )
    
#     data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    
#     data['theoretical_power'] = (
#         data['move_power'].fillna(0) * 
#         data['type_effectiveness'].fillna(1.0) * 
#         data['stab_multiplier'] * 
#         data['attack_defense_ratio'].fillna(1.0)
#     )
    
#     if 'previous_hp' in data.columns and 'max_hp' in data.columns:
#         data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
#     danger = np.ones(len(data))
#     danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
#     danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
#     danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
#     danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
#     data['weather_danger_level'] = danger
    
#     return data

# train_feat = engineer_features(train_df)
# test_feat = engineer_features(test_df)

# # 3. TARGET TRANSFORMATION (The Game Changer)
# # Drop rows in training where previous_hp is missing so our math doesn't break
# train_feat = train_feat.dropna(subset=['previous_hp', 'pikachu_hp'])

# # Create the new target: How much did the HP change this turn?
# train_feat['delta_hp'] = train_feat['pikachu_hp'] - train_feat['previous_hp']
# TARGET = 'delta_hp' 
# ID_COL = 'battle_turn'

# # These are the actual leaks present in both datasets
# LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied'] 

# y_train = train_feat[TARGET]

# # Drop ID, Target, Leaks, AND 'pikachu_hp' from the training set
# X_train = train_feat.drop(columns=[ID_COL, TARGET, 'pikachu_hp'] + LEAKED_FEATURES)

# # Drop ID and Leaks from the test set (using errors='ignore' just to be safe!)
# X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')

# # Impute missing max_hp and previous_hp in the test set to allow final math reconstruction
# test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())
# test_previous_hp = test_feat['previous_hp'].fillna(test_max_hp)

# # 4. Pipeline Setup
# categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
# numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# numeric_transformer = SimpleImputer(strategy='median')
# categorical_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='most_frequent')),
#     ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
# ])

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', numeric_transformer, numeric_cols),
#         ('cat', categorical_transformer, categorical_cols)
#     ])

# # 5. Train the Model on the Delta
# hgb_model = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', HistGradientBoostingRegressor(
#         max_iter=400,          # Boosted iterations slightly
#         learning_rate=0.04,    # Slowed down learning rate for precision
#         max_depth=7,           
#         random_state=42
#     ))
# ])

# print("Training model to predict damage dealt...")
# hgb_model.fit(X_train, y_train)

# # 6. Predict and Reconstruct Absolute HP
# print("Reconstructing final HP...")
# predicted_delta = hgb_model.predict(X_test)

# # Final HP = Previous HP + Predicted Change
# reconstructed_hp = test_previous_hp + predicted_delta

# # Apply the laws of physics (HP can't be less than 0 or greater than Max HP)
# final_predictions = np.clip(reconstructed_hp, 0, test_max_hp)

# submission = pd.DataFrame({
#     ID_COL: test_feat[ID_COL],
#     'pikachu_hp': final_predictions
# })

# submission.to_csv('submissionDay10.csv', index=False)
# print("Saved predictions to 'submissionDay10.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_25748\1607719605.py:78: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training model to predict damage dealt...
Reconstructing final HP...
Saved predictions to 'submissionDay10.csv'


In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import RandomizedSearchCV

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Feature Engineering Function (Cleaned up duplicates, kept ALL features)
def engineer_features(df):
    data = df.copy()
    
    # 2.1 Status Condition Adjustments
    data['adj_speed_pikachu'] = np.where(data['pikachu_status'] == 'Paralyzed', 
                                         data['speed_stat_pikachu'] * 0.5, 
                                         data['speed_stat_pikachu'])
    
    data['adj_attack_pikachu'] = np.where(data['pikachu_status'] == 'Burned', 
                                          data['attack_stat'] * 0.5, 
                                          data['attack_stat'])

    # 2.2 Stat Differentials (Using Adjusted Speed)
    data['speed_advantage'] = data['adj_speed_pikachu'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    # 2.3 Attack/Defense Ratios (Using Adjusted Attack)
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack_pikachu'] / (data['defense_stat'] + 1e-5)
    )
    
    # 2.4 STAB (Same Type Attack Bonus)
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    
    # 2.5 Theoretical Power
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['stab_multiplier'] * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    # 2.6 Relative HP state
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
    # 2.7 Weather Danger Level Mapping
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger

    return data

# Apply feature engineering
print("Engineering features...")
train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target and Remove Leaks
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)

# FILL MISSING MAX_HP to prevent NaNs during the clipping step
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Build Preprocessing Pipeline
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 5. Define Gradient Boosting Model (Base Pipeline)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(random_state=42))
])

# 6. The Tuning Grid (The Brute-Force Engine)
param_distributions = {
    'regressor__learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
    'regressor__max_iter': [300, 400, 500, 600],  # Tests more trees
    'regressor__max_depth': [5, 6, 7, 9, 12, None], 
    'regressor__max_leaf_nodes': [31, 50, 75, 120],
    'regressor__l2_regularization': [0.0, 0.1, 0.5, 1.0, 5.0]
}

print("Running Randomized Search (This will take a few minutes)...")
# Tests 30 random combinations of the parameters above, cross-validating 3 times each (90 total fits)
search = RandomizedSearchCV(
    pipeline, 
    param_distributions, 
    n_iter=30, 
    cv=3, 
    scoring='r2', 
    random_state=42, 
    n_jobs=-1
)

search.fit(X_train, y_train)

print(f"Best internal R2 score: {search.best_score_}")
print(f"Best parameters found:\n {search.best_params_}")

# 7. Predict and Post-Process using the absolute Best Model found
print("Predicting and applying physics constraints with the Champion Model...")
best_model = search.best_estimator_
raw_predictions = best_model.predict(X_test)

# Force predictions to be physically possible (0 to max_hp)
clipped_predictions = np.clip(raw_predictions, 0, test_max_hp)

# Create submission
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})

submission.to_csv('SubmissionDay10.csv', index=False)
print("Success! Saved clean predictions to 'SubmissionDay10_Tuned.csv'")

Engineering features...
Running Randomized Search (This will take a few minutes)...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_25748\324122292.py:83: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Best internal R2 score: 0.49555463106509573
Best parameters found:
 {'regressor__max_leaf_nodes': 75, 'regressor__max_iter': 500, 'regressor__max_depth': 9, 'regressor__learning_rate': 0.01, 'regressor__l2_regularization': 5.0}
Predicting and applying physics constraints with the Champion Model...
Success! Saved clean predictions to 'SubmissionDay10_Tuned.csv'


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Feature Engineering (Your exact winning setup)
def engineer_features(df):
    data = df.copy()
    data['adj_speed_pikachu'] = np.where(data['pikachu_status'] == 'Paralyzed', data['speed_stat_pikachu'] * 0.5, data['speed_stat_pikachu'])
    data['adj_attack_pikachu'] = np.where(data['pikachu_status'] == 'Burned', data['attack_stat'] * 0.5, data['attack_stat'])

    data['speed_advantage'] = data['adj_speed_pikachu'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack_pikachu'] / (data['defense_stat'] + 1e-5)
    )
    
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['theoretical_power'] = (
        data['move_power'].fillna(0) * 
        data['type_effectiveness'].fillna(1.0) * 
        data['stab_multiplier'] * 
        data['attack_defense_ratio'].fillna(1.0)
    )
    
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger

    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target and Remove Leaks
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)

test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Build Preprocessing Pipeline
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
# We still encode to integers so the model can read them, but we will tell the model they are categories!
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# --- NEW: Create a boolean mask to tell the model which columns are categories ---
# The ColumnTransformer outputs numeric columns first, then categorical columns
categorical_mask = [False] * len(numeric_cols) + [True] * len(categorical_cols)

# 5. Define Gradient Boosting Model (Unlocking Native Categorical Support)
hgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(
        categorical_features=categorical_mask, # <--- THIS IS THE MAGIC FIX
        max_iter=350, 
        learning_rate=0.05, 
        max_depth=6, 
        random_state=42
    ))
])

# Train
print("Training Gradient Boosting model with Native Categorical Splitting...")
hgb_model.fit(X_train, y_train)

# 6. Predict and Post-Process (Clipping)
print("Predicting and applying physics constraints...")
raw_predictions = hgb_model.predict(X_test)
clipped_predictions = np.clip(raw_predictions, 0, test_max_hp)

# Create submission
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})

submission.to_csv('submissionDay11.csv', index=False)
print("Success! Saved clean predictions to 'submissionDay11'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_12568\4273821339.py:65: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training Gradient Boosting model with Native Categorical Splitting...
Predicting and applying physics constraints...
Success! Saved clean predictions to 'submissionDay11'


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Feature Engineering (The 0.62889 base)
def engineer_features(df):
    data = df.copy()
    
    def calc_stage_multiplier(stage):
        return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
    data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
    data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
    data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    
    data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
    data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])

    data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack'] / (data['true_defense'] + 1e-5)
    )
    
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger

    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target (Executing your specific idea)
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'

# We are putting trainer_focus_score BACK on the drop list, 
# but keeping damage_dealt and healing_applied in the dataset!
LEAKED_FEATURES = ['trainer_focus_score'] 

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)
test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES)

# Fill missing test set values so the model has the data it needs
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())
X_test['damage_dealt'] = X_test['damage_dealt'].fillna(0)
X_test['healing_applied'] = X_test['healing_applied'].fillna(0)
X_test['previous_hp'] = X_test['previous_hp'].fillna(test_max_hp)

# 4. Preprocessing & Native Categoricals
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

categorical_mask = [False] * len(numeric_cols) + [True] * len(categorical_cols)

# 5. The Champion Model
hgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(
        categorical_features=categorical_mask, 
        max_iter=350, 
        learning_rate=0.05, 
        max_depth=6, 
        random_state=42
    ))
])

print("Training model with damage and healing features included...")
hgb_model.fit(X_train, y_train)

raw_predictions = hgb_model.predict(X_test)
clipped_predictions = np.clip(raw_predictions, 0, test_max_hp)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})

submission.to_csv('submissionDay12.csv', index=False)
print("Saved predictions to 'submissionDay12.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_11156\1687750647.py:74: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training model with damage and healing features included...
Saved predictions to 'submissionDay12.csv'


In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import cross_val_score, KFold

# 1. Load Training Data Only
print("Loading train.csv...")
train_df = pd.read_csv('train.csv')

# 2. Feature Engineering 
def engineer_features(df):
    data = df.copy()
    
    def calc_stage_multiplier(stage):
        return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
    data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
    data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
    data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    
    data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
    data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])

    data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack'] / (data['true_defense'] + 1e-5)
    )
    
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    
    if 'previous_hp' in data.columns and 'max_hp' in data.columns:
        data['previous_hp_ratio'] = data['previous_hp'] / (data['max_hp'] + 1e-5)
        
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger

    return data

print("Engineering features...")
train_feat = engineer_features(train_df)

# 3. Setup Target and Features
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'

# EXCLUDING ONLY TRAINER FOCUS
LEAKED_FEATURES = ['trainer_focus_score'] 

y = train_feat[TARGET]
X = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

# Ensure damage and healing NaNs are handled explicitly before CV
X['damage_dealt'] = X['damage_dealt'].fillna(0)
X['healing_applied'] = X['healing_applied'].fillna(0)

# 4. Preprocessing & Native Categoricals
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

categorical_mask = [False] * len(numeric_cols) + [True] * len(categorical_cols)

# 5. The Model Pipeline
hgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(
        categorical_features=categorical_mask, 
        max_iter=350, 
        learning_rate=0.05, 
        max_depth=6, 
        random_state=42
    ))
])

# 6. Local Evaluation (K-Fold Cross Validation)
print("\nRunning 5-Fold Cross Validation (Evaluating local R2 score)...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Calculate R2 for each fold
cv_scores = cross_val_score(hgb_model, X, y, cv=kf, scoring='r2', n_jobs=-1)

print("\n--- LOCAL EVALUATION RESULTS ---")
print(f"Scores for each fold: {cv_scores}")
print(f"Estimated Final R2 Score: {cv_scores.mean():.5f} (+/- {cv_scores.std():.5f})")
if cv_scores.mean() > 0.68:
    print("BOOM! You are mathematically projected to cross 0.68!")
else:
    print("Still under 0.68. We might need to keep digging.")

Loading train.csv...
Engineering features...

Running 5-Fold Cross Validation (Evaluating local R2 score)...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_12568\1508795987.py:70: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()



--- LOCAL EVALUATION RESULTS ---
Scores for each fold: [0.52925835 0.53666602 0.52017074 0.53124181 0.51700763]
Estimated Final R2 Score: 0.52687 (+/- 0.00725)
Still under 0.68. We might need to keep digging.


In [ ]:
# import pandas as pd    ## second best model uptill now
# import numpy as np
# from sklearn.ensemble import HistGradientBoostingRegressor
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OrdinalEncoder
# from sklearn.model_selection import KFold

# # 1. Load Data
# train_df = pd.read_csv('train.csv')
# test_df = pd.read_csv('test.csv')

# # 2. Engineer Core & Leak-Based Features
# def engineer_features(df):
#     data = df.copy()
    
#     # --- NEW: The Deterministic Leak Features ---
#     # Fill missing values locally just for this math calculation
#     safe_damage = data['damage_dealt'].fillna(0)
#     safe_healing = data['healing_applied'].fillna(0)
    
#     # If max_hp is available use it as a fallback, otherwise assume 100
#     if 'max_hp' in data.columns:
#         safe_prev_hp = data['previous_hp'].fillna(data['max_hp'])
#     else:
#         safe_prev_hp = data['previous_hp'].fillna(100) 
        
#     data['net_hp_change'] = safe_healing - safe_damage
#     # This directly hands the model the exact mathematical answer
#     data['projected_hp'] = safe_prev_hp + data['net_hp_change']
    
#     # --- The Game Mechanics (Your 0.62889 Base) ---
#     def calc_stage_multiplier(stage):
#         return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
#     data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
#     data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
#     data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    
#     data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
#     data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])

#     data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
#     data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
#     data['attack_defense_ratio'] = np.where(
#         data['move_category'] == 'Special',
#         data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
#         data['adj_attack'] / (data['true_defense'] + 1e-5)
#     )
    
#     data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
#     data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    
#     data['battle_fatigue'] = data['round'].fillna(1) * data['turn'].fillna(1)
    
#     return data

# train_feat = engineer_features(train_df)
# test_feat = engineer_features(test_df)

# # 3. Target and Leak Handling
# TARGET = 'pikachu_hp'
# ID_COL = 'battle_turn'

# # We keep damage_dealt and healing_applied active, drop only trainer_focus_score!
# LEAKED_FEATURES = ['trainer_focus_score']

# y_train = train_feat[TARGET]
# X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)
# test_ids = test_feat[ID_COL]
# X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')

# test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# # 4. Preprocessing 
# categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
# numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# numeric_transformer = SimpleImputer(strategy='median')
# categorical_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='most_frequent')),
#     ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
# ])

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', numeric_transformer, numeric_cols),
#         ('cat', categorical_transformer, categorical_cols)
#     ])

# categorical_mask = [False] * len(numeric_cols) + [True] * len(categorical_cols)

# # 5. K-Fold Ensembling with Leak-Aware Model
# N_SPLITS = 5
# kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# test_predictions = np.zeros(len(X_test))

# print(f"Training {N_SPLITS}-Fold Ensemble with Engineered Leak Features...")

# for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
#     print(f"--- Training Fold {fold + 1}/{N_SPLITS} ---")
    
#     X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    
#     # Mild l2_regularization applied to prevent the trees from totally collapsing onto the projected_hp column
#     model = Pipeline(steps=[
#         ('preprocessor', preprocessor),
#         ('regressor', HistGradientBoostingRegressor(
#             categorical_features=categorical_mask,
#             max_iter=400, 
#             learning_rate=0.04, 
#             max_depth=6, 
#             l2_regularization=0.1, 
#             random_state=42 + fold
#         ))
#     ])
    
#     model.fit(X_tr, y_tr)
#     test_predictions += model.predict(X_test) / N_SPLITS

# # 6. Post-Process (Clipping)
# print("\nApplying physics constraints...")
# clipped_predictions = np.clip(test_predictions, 0, test_max_hp)

# submission = pd.DataFrame({
#     ID_COL: test_ids,
#     TARGET: clipped_predictions
# })

# submission.to_csv('submissionDay12.csv', index=False)
# print("Success! Saved predictions to 'submissionDay12.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_11156\1237439034.py:78: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training 5-Fold Ensemble with Engineered Leak Features...
--- Training Fold 1/5 ---
--- Training Fold 2/5 ---
--- Training Fold 3/5 ---
--- Training Fold 4/5 ---
--- Training Fold 5/5 ---

Applying physics constraints...
Success! Saved predictions to 'submissionDay12.csv'


In [ ]:
# import pandas as pd  0.65542
# import numpy as np
# from sklearn.ensemble import HistGradientBoostingRegressor
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OrdinalEncoder
# from sklearn.model_selection import KFold

# # 1. Load Data
# train_df = pd.read_csv('train.csv')
# test_df = pd.read_csv('test.csv')

# # 2. Engineer Core & Leak-Based Features
# def engineer_features(df):
#     data = df.copy()
    
#     # Deterministic Leak Features
#     safe_damage = data['damage_dealt'].fillna(0)
#     safe_healing = data['healing_applied'].fillna(0)
    
#     if 'max_hp' in data.columns:
#         safe_prev_hp = data['previous_hp'].fillna(data['max_hp'])
#     else:
#         safe_prev_hp = data['previous_hp'].fillna(100) 
        
#     data['net_hp_change'] = safe_healing - safe_damage
#     data['projected_hp'] = safe_prev_hp + data['net_hp_change']
    
#     # Game Mechanics
#     def calc_stage_multiplier(stage):
#         return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
#     data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
#     data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
#     data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    
#     data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
#     data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])

#     data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
#     data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
#     data['attack_defense_ratio'] = np.where(
#         data['move_category'] == 'Special',
#         data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
#         data['adj_attack'] / (data['true_defense'] + 1e-5)
#     )
    
#     data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
#     data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    
#     data['battle_fatigue'] = data['round'].fillna(1) * data['turn'].fillna(1)
    
#     return data

# train_feat = engineer_features(train_df)
# test_feat = engineer_features(test_df)

# # 3. Target Transformation: Predicting the Residual
# ID_COL = 'battle_turn'
# LEAKED_FEATURES = ['trainer_focus_score']

# # Failsafe: Ensure our mathematical baseline didn't create NaNs 
# train_feat['projected_hp'] = train_feat['projected_hp'].fillna(train_feat['pikachu_hp'])

# # The model must learn the difference between reality and our math formula
# train_feat['residual'] = train_feat['pikachu_hp'] - train_feat['projected_hp']
# TARGET = 'residual'

# # CRITICAL FIX: Drop any training rows where the target is NaN
# train_feat = train_feat.dropna(subset=[TARGET, 'pikachu_hp'])

# # We still drop the original pikachu_hp so it doesn't cheat!
# y_train = train_feat[TARGET]
# X_train = train_feat.drop(columns=[TARGET, 'pikachu_hp', ID_COL] + LEAKED_FEATURES)

# test_ids = test_feat[ID_COL]
# X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')
# test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# # 4. Preprocessing 
# categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
# numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# numeric_transformer = SimpleImputer(strategy='median')
# categorical_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='most_frequent')),
#     ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
# ])

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', numeric_transformer, numeric_cols),
#         ('cat', categorical_transformer, categorical_cols)
#     ])

# categorical_mask = [False] * len(numeric_cols) + [True] * len(categorical_cols)

# # 5. K-Fold Ensembling on the Residual
# N_SPLITS = 5
# kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# # Array to store the predicted residuals
# test_residuals = np.zeros(len(X_test))

# print(f"Training {N_SPLITS}-Fold Ensemble to predict residuals...")

# for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
#     X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    
#     model = Pipeline(steps=[
#         ('preprocessor', preprocessor),
#         ('regressor', HistGradientBoostingRegressor(
#             categorical_features=categorical_mask,
#             max_iter=400, 
#             learning_rate=0.04, 
#             max_depth=6, 
#             l2_regularization=0.1, 
#             random_state=42 + fold
#         ))
#     ])
    
#     model.fit(X_tr, y_tr)
#     test_residuals += model.predict(X_test) / N_SPLITS

# # 6. Reconstruct the Final Answer
# print("\nReconstructing HP and applying physical bounds...")

# # Failsafe 1: If both previous_hp and max_hp were missing, force the projection to 100
# safe_test_projected = X_test['projected_hp'].fillna(100)

# # Final Answer = Our Deterministic Math + The Model's Predicted Corrections
# final_predictions = safe_test_projected + test_residuals

# clipped_predictions = np.clip(final_predictions, 0, test_max_hp)

# submission = pd.DataFrame({
#     ID_COL: test_ids,
#     'pikachu_hp': clipped_predictions 
# })

# # Failsafe 2: The ultimate guarantee that absolutely NO NaNs slip into Kaggle
# submission['pikachu_hp'] = submission['pikachu_hp'].fillna(0)

# submission.to_csv('submissionDay12.csv', index=False)
# print("Success! Saved predictions to 'submissionDay12.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_11156\273212261.py:83: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training 5-Fold Ensemble to predict residuals...

Reconstructing HP and applying physical bounds...
Success! Saved predictions to 'submissionDay12.csv'


In [ ]:
import pandas as pd #best model
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import KFold

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Advanced Feature Engineering
def engineer_features(df):
    data = df.copy()
    
    # --- The 0.656 Deterministic Math ---
    safe_damage = data['damage_dealt'].fillna(0)
    safe_healing = data['healing_applied'].fillna(0)
    
    if 'max_hp' in data.columns:
        safe_prev_hp = data['previous_hp'].fillna(data['max_hp'])
    else:
        safe_prev_hp = data['previous_hp'].fillna(100) 
        
    data['net_hp_change'] = safe_healing - safe_damage
    data['projected_hp'] = safe_prev_hp + data['net_hp_change']
    
    # --- Core Game Mechanics ---
    def calc_stage_multiplier(stage):
        return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
    data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
    data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
    data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    
    data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
    data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])

    data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack'] / (data['true_defense'] + 1e-5)
    )
    
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    data['battle_fatigue'] = data['round'].fillna(1) * data['turn'].fillna(1)

    # (Paste this right above the NEW Categorical Synergies block)
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger

    # --- NEW: Categorical Synergies & Momentum ---
    data['attacks_first'] = np.where(data['speed_advantage'] > 0, 1, 0)
    
    # Concatenating strings forces the Native Categorical engine to learn the unique matchups
    data['weather_move_synergy'] = data['weather_condition'].astype(str) + "_" + data['move_type'].astype(str)
    data['type_matchup_str'] = data['move_type'].astype(str) + "_vs_" + data['opponent_type'].astype(str)

    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Standard Target Setup
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)
test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')

test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Pipeline & Native Categoricals
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

categorical_mask = [False] * len(numeric_cols) + [True] * len(categorical_cols)

# 5. K-Fold Ensembling (Restored Normal Prediction)
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

test_predictions = np.zeros(len(X_test))

print("Training Advanced Synergy 5-Fold Ensemble...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    
    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', HistGradientBoostingRegressor(
            categorical_features=categorical_mask,
            max_iter=400, 
            learning_rate=0.04, 
            max_depth=6, 
            l2_regularization=0.1, 
            random_state=42 + fold
        ))
    ])
    
    model.fit(X_tr, y_tr)
    test_predictions += model.predict(X_test) / N_SPLITS

# 6. Apply Bounds & Submit
print("Clipping predictions to valid physical bounds...")
clipped_predictions = np.clip(test_predictions, 0, test_max_hp)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})

# Absolute failsafe
submission[TARGET] = submission[TARGET].fillna(0)
submission.to_csv('submissionDay12.csv', index=False)
print("Saved predictions to 'submissionDay12.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_11156\2483463129.py:87: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training Advanced Synergy 5-Fold Ensemble...
Clipping predictions to valid physical bounds...
Saved predictions to 'submissionDay12.csv'


In [7]:
import pandas as pd #best model
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import KFold

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Advanced Feature Engineering
def engineer_features(df):
    data = df.copy()
    
    # --- The 0.656 Deterministic Math ---
    safe_damage = data['damage_dealt'].fillna(0)
    safe_healing = data['healing_applied'].fillna(0)
    
    if 'max_hp' in data.columns:
        safe_prev_hp = data['previous_hp'].fillna(data['max_hp'])
    else:
        safe_prev_hp = data['previous_hp'].fillna(100) 
        
    data['net_hp_change'] = safe_healing - safe_damage
    data['projected_hp'] = safe_prev_hp + data['net_hp_change']
    
    # --- Core Game Mechanics ---
    def calc_stage_multiplier(stage):
        return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
    data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
    data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
    data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    
    data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
    data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])

    data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack'] / (data['true_defense'] + 1e-5)
    )
    
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    data['battle_fatigue'] = data['round'].fillna(1) * data['turn'].fillna(1)
    
    # --- Environment & Weather Multipliers ---
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger
    
    # --- Categorical Synergies & Momentum ---
    data['attacks_first'] = np.where(data['speed_advantage'] > 0, 1, 0)
    data['weather_move_synergy'] = data['weather_condition'].astype(str) + "_" + data['move_type'].astype(str)
    data['type_matchup_str'] = data['move_type'].astype(str) + "_vs_" + data['opponent_type'].astype(str)

    # --- NEW: Continuous Numerical Features ---
    # Polynomial interaction for raw force
    data['raw_kinetic_energy'] = data['move_power'].fillna(0) * data['true_attack']
    
    # Log transformation for the multiplicative skew
    data['log_attack_defense_ratio'] = np.log1p(data['attack_defense_ratio'])
    
    # Numerical wound tracking
    if 'max_hp' in data.columns and 'previous_hp' in data.columns:
        data['hp_deficit'] = data['max_hp'] - data['previous_hp']
    else:
        data['hp_deficit'] = 0
        
    # Level-scaled physical advantage
    data['level_scaled_attack'] = data['true_attack'] * data['pikachu_level']

    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Standard Target Setup
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)
test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')

test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Pipeline & Native Categoricals
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

categorical_mask = [False] * len(numeric_cols) + [True] * len(categorical_cols)

# 5. K-Fold Ensembling (Restored Normal Prediction)
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

test_predictions = np.zeros(len(X_test))

print("Training Advanced Synergy 5-Fold Ensemble...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    
    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', HistGradientBoostingRegressor(
            categorical_features=categorical_mask,
            max_iter=400, 
            learning_rate=0.04, 
            max_depth=6, 
            l2_regularization=0.1, 
            random_state=42 + fold
        ))
    ])
    
    model.fit(X_tr, y_tr)
    test_predictions += model.predict(X_test) / N_SPLITS

# 6. Apply Bounds & Submit
print("Clipping predictions to valid physical bounds...")
clipped_predictions = np.clip(test_predictions, 0, test_max_hp)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})

# Absolute failsafe
submission[TARGET] = submission[TARGET].fillna(0)
submission.to_csv('submissionDay12.csv', index=False)
print("Saved predictions to 'submissionDay12.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_11156\2933005554.py:101: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training Advanced Synergy 5-Fold Ensemble...
Clipping predictions to valid physical bounds...
Saved predictions to 'submissionDay12.csv'


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import RandomizedSearchCV, KFold
from scipy.stats import uniform, randint

# 1. Load Data
print("Loading datasets...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Your Champion Feature Engineering (0.6634 Base)
def engineer_features(df):
    data = df.copy()
    
    safe_damage = data['damage_dealt'].fillna(0)
    safe_healing = data['healing_applied'].fillna(0)
    
    if 'max_hp' in data.columns:
        safe_prev_hp = data['previous_hp'].fillna(data['max_hp'])
    else:
        safe_prev_hp = data['previous_hp'].fillna(100) 
        
    data['net_hp_change'] = safe_healing - safe_damage
    data['projected_hp'] = safe_prev_hp + data['net_hp_change']
    
    def calc_stage_multiplier(stage):
        return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
    data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
    data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
    data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    
    data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
    data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])

    data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack'] / (data['true_defense'] + 1e-5)
    )
    
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    data['battle_fatigue'] = data['round'].fillna(1) * data['turn'].fillna(1)
    
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger
    
    data['attacks_first'] = np.where(data['speed_advantage'] > 0, 1, 0)
    data['weather_move_synergy'] = data['weather_condition'].astype(str) + "_" + data['move_type'].astype(str)
    data['type_matchup_str'] = data['move_type'].astype(str) + "_vs_" + data['opponent_type'].astype(str)

    data['raw_kinetic_energy'] = data['move_power'].fillna(0) * data['true_attack']
    data['log_attack_defense_ratio'] = np.log1p(data['attack_defense_ratio'])
    
    if 'max_hp' in data.columns and 'previous_hp' in data.columns:
        data['hp_deficit'] = data['max_hp'] - data['previous_hp']
    else:
        data['hp_deficit'] = 0
        
    data['level_scaled_attack'] = data['true_attack'] * data['pikachu_level']

    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target and Inputs
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)
test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')

test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Preprocessing & Native Categoricals
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

categorical_mask = [False] * len(numeric_cols) + [True] * len(categorical_cols)

# 5. Base Model Pipeline
base_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(
        categorical_features=categorical_mask,
        random_state=42
    ))
])

# 6. Hyperparameter Search Space
param_distributions = {
    'regressor__learning_rate': uniform(0.02, 0.08),      # 0.02 to 0.10
    'regressor__max_iter': randint(300, 600),            # 300 to 600 trees
    'regressor__max_depth': randint(5, 9),               # 5 to 8 depth
    'regressor__l2_regularization': uniform(0.01, 1.0),   # Regularization penalty
    'regressor__min_samples_leaf': randint(15, 40)
}

# 7. Search & Auto-Refit
print("Starting search across combinations...")
kf = KFold(n_splits=3, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    base_model, 
    param_distributions=param_distributions, 
    n_iter=25,
    cv=kf, 
    scoring='r2', 
    n_jobs=-1,
    random_state=42,
    refit=True  # Automatically refits the best model on all training data
)

search.fit(X_train, y_train)

print(f"\nOptimization complete! Best Local CV R2: {search.best_score_:.5f}")

# 8. Predict on Test Set & Save Submission
best_model = search.best_estimator_
raw_preds = best_model.predict(X_test)
clipped_preds = np.clip(raw_preds, 0, test_max_hp)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_preds
})

submission[TARGET] = submission[TARGET].fillna(0)
submission.to_csv('submissionDay13.csv', index=False)
print("Saved predictions to 'submissionDay13.csv'")

Loading datasets...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_15580\2141745347.py:93: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Starting search across combinations...

Optimization complete! Best Local CV R2: 0.53124
Saved predictions to 'submissionDay13.csv'


In [2]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import KFold
from xgboost import XGBRegressor  # THE NEW ENGINE

# 1. Load Data
print("Loading data...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Champion Feature Engineering (Unchanged)
def engineer_features(df):
    data = df.copy()
    
    safe_damage = data['damage_dealt'].fillna(0)
    safe_healing = data['healing_applied'].fillna(0)
    
    if 'max_hp' in data.columns:
        safe_prev_hp = data['previous_hp'].fillna(data['max_hp'])
    else:
        safe_prev_hp = data['previous_hp'].fillna(100) 
        
    data['net_hp_change'] = safe_healing - safe_damage
    data['projected_hp'] = safe_prev_hp + data['net_hp_change']
    
    def calc_stage_multiplier(stage):
        return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
    data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
    data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
    data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    
    data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
    data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])

    data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack'] / (data['true_defense'] + 1e-5)
    )
    
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    data['battle_fatigue'] = data['round'].fillna(1) * data['turn'].fillna(1)
    
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger
    
    data['attacks_first'] = np.where(data['speed_advantage'] > 0, 1, 0)
    data['weather_move_synergy'] = data['weather_condition'].astype(str) + "_" + data['move_type'].astype(str)
    data['type_matchup_str'] = data['move_type'].astype(str) + "_vs_" + data['opponent_type'].astype(str)

    data['raw_kinetic_energy'] = data['move_power'].fillna(0) * data['true_attack']
    data['log_attack_defense_ratio'] = np.log1p(data['attack_defense_ratio'])
    
    if 'max_hp' in data.columns and 'previous_hp' in data.columns:
        data['hp_deficit'] = data['max_hp'] - data['previous_hp']
    else:
        data['hp_deficit'] = 0
        
    data['level_scaled_attack'] = data['true_attack'] * data['pikachu_level']

    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)
test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. XGBoost Categorical Encoding
# XGBoost can natively handle strings if they are cast to the 'category' dtype
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# 5. K-Fold Ensembling with XGBoost
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
test_predictions = np.zeros(len(X_test))

print("Training XGBoost 5-Fold Ensemble...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f" - Fold {fold + 1}...")
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    
    # We configure XGBoost to use its fast histogram mode and enable categorical support
    model = XGBRegressor(
        tree_method='hist',
        enable_categorical=True,
        n_estimators=450,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.5,      # Strong L2 Regularization
        reg_alpha=0.5,       # Strong L1 Regularization
        random_state=42 + fold
    )
    
    model.fit(X_tr, y_tr)
    test_predictions += model.predict(X_test) / N_SPLITS

# 6. Apply Bounds & Export
print("Clipping predictions to valid physical bounds...")
clipped_predictions = np.clip(test_predictions, 0, test_max_hp)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})

submission[TARGET] = submission[TARGET].fillna(0)
submission.to_csv('submissionDay13.csv', index=False)
print("Success! Saved to 'submissionDay13.csv'")

Loading data...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_15580\888523085.py:92: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()


Training XGBoost 5-Fold Ensemble...
 - Fold 1...
 - Fold 2...
 - Fold 3...
 - Fold 4...
 - Fold 5...
Clipping predictions to valid physical bounds...
Success! Saved to 'submissionDay13.csv'


In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

# 1. Load Data
print("Loading data...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Champion Feature Engineering (Advanced Numerical Master)
def engineer_features(df):
    data = df.copy()
    
    safe_damage = data['damage_dealt'].fillna(0)
    safe_healing = data['healing_applied'].fillna(0)
    safe_prev_hp = data['previous_hp'].fillna(data['max_hp'] if 'max_hp' in data.columns else 100)
    data['net_hp_change'] = safe_healing - safe_damage
    data['projected_hp'] = safe_prev_hp + data['net_hp_change']
    
    def calc_stage_multiplier(stage):
        return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
    data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
    data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
    data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    
    data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
    data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])

    data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack'] / (data['true_defense'] + 1e-5)
    )
    
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    
    data['battle_fatigue'] = data['round'].fillna(1) * data['turn'].fillna(1)
    data['exponential_fatigue'] = data['battle_fatigue'] ** 2
    
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger
    
    data['attacks_first'] = np.where(data['speed_advantage'] > 0, 1, 0)
    data['weather_move_synergy'] = data['weather_condition'].astype(str) + "_" + data['move_type'].astype(str)
    data['type_matchup_str'] = data['move_type'].astype(str) + "_vs_" + data['opponent_type'].astype(str)

    data['raw_kinetic_energy'] = data['move_power'].fillna(0) * data['true_attack']
    data['log_attack_defense_ratio'] = np.log1p(data['attack_defense_ratio'])
    
    data['hp_deficit'] = data['max_hp'] - data['previous_hp'] if 'max_hp' in data.columns else 0
    data['level_scaled_attack'] = data['true_attack'] * data['pikachu_level']

    data['speed_ratio'] = data['adj_speed'] / (data['speed_stat_opponent'] + 1e-5)
    data['stat_product'] = data['true_attack'] * data['true_defense'] * data['adj_speed']
    
    level_factor = (2 * data['pikachu_level'] / 5) + 2
    power = data['move_power'].fillna(0)
    data['core_base_damage'] = ((level_factor * power * data['attack_defense_ratio']) / 50) + 2

    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)
test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Data Preprocessing (Creating two versions for the two models)
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor_hgb = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])
categorical_mask = [False] * len(numeric_cols) + [True] * len(categorical_cols)

X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()
for col in categorical_cols:
    X_train_xgb[col] = X_train_xgb[col].astype('category')
    X_test_xgb[col] = X_test_xgb[col].astype('category')

# 5. Stacking Setup: Out-Of-Fold Arrays
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Arrays to store the Meta-Model training data (OOF predictions)
oof_hgb = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))

# Arrays to store the Base Model test predictions
test_preds_hgb = np.zeros(len(X_test))
test_preds_xgb = np.zeros(len(X_test))

print("Training Level 1 Base Models to generate Meta-Features...")

# 6. Train Base Models and Collect OOF Predictions
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f" - Training Fold {fold + 1}/{N_SPLITS}...")
    
    # --- HistGradient Training ---
    X_tr_hgb, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val_hgb = X_train.iloc[val_idx]
    
    model_hgb = Pipeline(steps=[
        ('preprocessor', preprocessor_hgb),
        ('regressor', HistGradientBoostingRegressor(
            categorical_features=categorical_mask,
            max_iter=400, learning_rate=0.04, max_depth=6, l2_regularization=0.1, random_state=42 + fold
        ))
    ])
    model_hgb.fit(X_tr_hgb, y_tr)
    oof_hgb[val_idx] = model_hgb.predict(X_val_hgb)
    test_preds_hgb += model_hgb.predict(X_test) / N_SPLITS
    
    # --- XGBoost Training ---
    X_tr_xgb = X_train_xgb.iloc[train_idx]
    X_val_xgb = X_train_xgb.iloc[val_idx]
    
    model_xgb = XGBRegressor(
        tree_method='hist', enable_categorical=True, n_estimators=450, learning_rate=0.03, 
        max_depth=6, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.5, reg_alpha=0.5, random_state=42 + fold
    )
    model_xgb.fit(X_tr_xgb, y_tr)
    oof_xgb[val_idx] = model_xgb.predict(X_val_xgb)
    test_preds_xgb += model_xgb.predict(X_test_xgb) / N_SPLITS

# 7. Level 2: Train the Meta-Model (Ridge Regression)
print("\nTraining Level 2 Meta-Model (Ridge Regression) on OOF predictions...")
X_meta_train = np.column_stack((oof_hgb, oof_xgb))
X_meta_test = np.column_stack((test_preds_hgb, test_preds_xgb))

# Ridge regression acts as our Meta-Model, using L2 penalty to safely combine the two signals
meta_model = Ridge(alpha=1.0, random_state=42)
meta_model.fit(X_meta_train, y_train)

print("Meta-Model weights (HistGradient vs XGBoost):", meta_model.coef_)

# 8. Final Meta-Model Predictions
final_stack_preds = meta_model.predict(X_meta_test)

print("Clipping predictions to physical boundaries...")
clipped_predictions = np.clip(final_stack_preds, 0, test_max_hp)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})
submission[TARGET] = submission[TARGET].fillna(0)
submission.to_csv('submissionDay13.csv', index=False)
print("Success! Saved to 'submissionDay13.csv'")

Loading data...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_15580\4077994377.py:93: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training Level 1 Base Models to generate Meta-Features...
 - Training Fold 1/5...
 - Training Fold 2/5...
 - Training Fold 3/5...
 - Training Fold 4/5...
 - Training Fold 5/5...

Training Level 2 Meta-Model (Ridge Regression) on OOF predictions...
Meta-Model weights (HistGradient vs XGBoost): [0.41825096 0.58628229]
Clipping predictions to physical boundaries...
Success! Saved to 'submissionDay13.csv'


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

# 1. Load Data
print("Loading data for the final run...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

def engineer_features(df):
    data = df.copy()
    
    # --- 1. Core Mechanics (Needed for Damage Calculation) ---
    def calc_stage_multiplier(stage):
        return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
    data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
    data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
    data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    
    data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
    data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])

    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack'] / (data['true_defense'] + 1e-5)
    )
    
    # --- 2. THE GOLDEN IMPUTATION ---
    # Calculate expected damage first
    level_factor = (2 * data['pikachu_level'] / 5) + 2
    power = data['move_power'].fillna(0)
    data['core_base_damage'] = ((level_factor * power * data['attack_defense_ratio']) / 50) + 2
    
    # Use our math formula to fill missing damage instead of 0!
    smart_damage = data['damage_dealt'].fillna(data['core_base_damage'])
    safe_healing = data['healing_applied'].fillna(0)
    
    safe_prev_hp = data['previous_hp'].fillna(data['max_hp'] if 'max_hp' in data.columns else 100)
    data['net_hp_change'] = safe_healing - smart_damage
    data['projected_hp'] = safe_prev_hp + data['net_hp_change']
    
    # --- 3. The Remaining Features ---
    data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    
    data['battle_fatigue'] = data['round'].fillna(1) * data['turn'].fillna(1)
    data['exponential_fatigue'] = data['battle_fatigue'] ** 2
    
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger
    
    data['attacks_first'] = np.where(data['speed_advantage'] > 0, 1, 0)
    data['weather_move_synergy'] = data['weather_condition'].astype(str) + "_" + data['move_type'].astype(str)
    data['type_matchup_str'] = data['move_type'].astype(str) + "_vs_" + data['opponent_type'].astype(str)

    data['raw_kinetic_energy'] = data['move_power'].fillna(0) * data['true_attack']
    data['log_attack_defense_ratio'] = np.log1p(data['attack_defense_ratio'])
    
    data['hp_deficit'] = data['max_hp'] - data['previous_hp'] if 'max_hp' in data.columns else 0
    data['level_scaled_attack'] = data['true_attack'] * data['pikachu_level']
    data['speed_ratio'] = data['adj_speed'] / (data['speed_stat_opponent'] + 1e-5)
    data['stat_product'] = data['true_attack'] * data['true_defense'] * data['adj_speed']

    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# Setup Target
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)
test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# XGBoost Categorical Encoding
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# K-Fold Training
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
test_predictions = np.zeros(len(X_test))

print("Training Final XGBoost Model...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    
    model = XGBRegressor(
        tree_method='hist',
        enable_categorical=True,
        n_estimators=450,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.5,      
        reg_alpha=0.5,       
        random_state=42 + fold
    )
    
    model.fit(X_tr, y_tr)
    test_predictions += model.predict(X_test) / N_SPLITS

# Apply Bounds & INT POST-PROCESSING
print("Applying strict integer bounds...")
clipped_predictions = np.clip(test_predictions, 0, test_max_hp)

# THE METRIC HACK: Force to exact integer
final_integer_preds = np.round(clipped_predictions)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: final_integer_preds
})

submission[TARGET] = submission[TARGET].fillna(0)
submission.to_csv('submissionDay14.csv', index=False)
print("Saved to 'submissionDay14.csv'")

Loading data for the final run...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_32020\3313172554.py:90: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training Final XGBoost Model...
Applying strict integer bounds...
Saved to 'submissionDay14.csv'


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from scipy.optimize import minimize
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Feature Engineering (The Clean Math Baseline)
def engineer_features(df):
    data = df.copy()
    
    data['damage_is_missing'] = data['damage_dealt'].isna().astype(int)
    data['healing_is_missing'] = data['healing_applied'].isna().astype(int)
    
    safe_damage = data['damage_dealt'].fillna(0)
    safe_healing = data['healing_applied'].fillna(0)
    safe_prev_hp = data['previous_hp'].fillna(data['max_hp'] if 'max_hp' in data.columns else 100)
    
    data['net_hp_change'] = safe_healing - safe_damage
    data['projected_hp'] = safe_prev_hp + data['net_hp_change']
    
    def calc_stage_multiplier(stage):
        return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
    data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
    data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
    data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    
    data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
    data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])

    data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack'] / (data['true_defense'] + 1e-5)
    )
    
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    
    data['battle_fatigue'] = data['round'].fillna(1) * data['turn'].fillna(1)
    data['exponential_fatigue'] = data['battle_fatigue'] ** 2
    
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger
    
    data['attacks_first'] = np.where(data['speed_advantage'] > 0, 1, 0)
    data['weather_move_synergy'] = data['weather_condition'].astype(str) + "_" + data['move_type'].astype(str)
    data['type_matchup_str'] = data['move_type'].astype(str) + "_vs_" + data['opponent_type'].astype(str)

    data['raw_kinetic_energy'] = data['move_power'].fillna(0) * data['true_attack']
    data['log_attack_defense_ratio'] = np.log1p(data['attack_defense_ratio'])
    
    data['hp_deficit'] = data['max_hp'] - data['previous_hp'] if 'max_hp' in data.columns else 0
    data['level_scaled_attack'] = data['true_attack'] * data['pikachu_level']

    data['speed_ratio'] = data['adj_speed'] / (data['speed_stat_opponent'] + 1e-5)
    data['stat_product'] = data['true_attack'] * data['true_defense'] * data['adj_speed']
    
    level_factor = (2 * data['pikachu_level'] / 5) + 2
    power = data['move_power'].fillna(0)
    data['core_base_damage'] = ((level_factor * power * data['attack_defense_ratio']) / 50) + 2

    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)
test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Universal Categorical Encoding (Works for both XGB and LGBM!)
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# 5. Out-of-Fold 5-Fold Training
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_lgbm = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
test_preds_lgbm = np.zeros(len(X_test))
test_preds_xgb = np.zeros(len(X_test))

print("Training Level-1 Base Models (LightGBM + XGBoost)...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    
    # --- LightGBM Model ---
    model_lgbm = LGBMRegressor(
        n_estimators=450,
        learning_rate=0.03,
        max_depth=8,
        num_leaves=64,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42 + fold,
        verbose=-1
    )
    model_lgbm.fit(X_tr, y_tr)
    oof_lgbm[val_idx] = model_lgbm.predict(X_val)
    test_preds_lgbm += model_lgbm.predict(X_test) / N_SPLITS
    
    # --- XGBoost Model ---
    model_xgb = XGBRegressor(
        tree_method='hist', enable_categorical=True, n_estimators=450, learning_rate=0.03, 
        max_depth=6, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.5, reg_alpha=0.5, random_state=42 + fold
    )
    model_xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = model_xgb.predict(X_val)
    test_preds_xgb += model_xgb.predict(X_test) / N_SPLITS

# 6. Constrained Optimization (Non-Negative Stacking)
print("Finding optimal meta-weights minimizing MSE...")

def loss_func(weights):
    w1, w2 = weights
    pred = w1 * oof_lgbm + w2 * oof_xgb
    return np.mean((y_train - pred) ** 2)

initial_weights = [0.5, 0.5]
bounds = [(0, 1), (0, 1)]
constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})

opt_res = minimize(loss_func, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
best_w1, best_w2 = opt_res.x
print(f"Optimal Weights: LightGBM = {best_w1:.4f}, XGBoost = {best_w2:.4f}")

# 7. Generate Continuous Predictions (No Rounding)
final_preds = (best_w1 * test_preds_lgbm) + (best_w2 * test_preds_xgb)
clipped_predictions = np.clip(final_preds, 0, test_max_hp)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})
submission[TARGET] = submission[TARGET].fillna(0)
submission.to_csv('submissionDay14.csv', index=False)
print("Saved predictions to 'submissionDay14.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_27892\556531390.py:92: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training Level-1 Base Models (LightGBM + XGBoost)...
Finding optimal meta-weights minimizing MSE...
Optimal Weights: LightGBM = 0.8768, XGBoost = 0.1232
Saved predictions to 'submissionDay14.csv'


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Champion Feature Engineering
def engineer_features(df):
    data = df.copy()
    
    data['damage_is_missing'] = data['damage_dealt'].isna().astype(int)
    data['healing_is_missing'] = data['healing_applied'].isna().astype(int)
    
    safe_damage = data['damage_dealt'].fillna(0)
    safe_healing = data['healing_applied'].fillna(0)
    safe_prev_hp = data['previous_hp'].fillna(data['max_hp'] if 'max_hp' in data.columns else 100)
    
    data['net_hp_change'] = safe_healing - safe_damage
    data['projected_hp'] = safe_prev_hp + data['net_hp_change']
    
    def calc_stage_multiplier(stage):
        return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
    data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
    data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
    data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    
    data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
    data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])

    data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack'] / (data['true_defense'] + 1e-5)
    )
    
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    
    data['battle_fatigue'] = data['round'].fillna(1) * data['turn'].fillna(1)
    data['exponential_fatigue'] = data['battle_fatigue'] ** 2
    
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger
    
    data['attacks_first'] = np.where(data['speed_advantage'] > 0, 1, 0)
    data['weather_move_synergy'] = data['weather_condition'].astype(str) + "_" + data['move_type'].astype(str)
    data['type_matchup_str'] = data['move_type'].astype(str) + "_vs_" + data['opponent_type'].astype(str)

    data['raw_kinetic_energy'] = data['move_power'].fillna(0) * data['true_attack']
    data['log_attack_defense_ratio'] = np.log1p(data['attack_defense_ratio'])
    
    data['hp_deficit'] = data['max_hp'] - data['previous_hp'] if 'max_hp' in data.columns else 0
    data['level_scaled_attack'] = data['true_attack'] * data['pikachu_level']

    data['speed_ratio'] = data['adj_speed'] / (data['speed_stat_opponent'] + 1e-5)
    data['stat_product'] = data['true_attack'] * data['true_defense'] * data['adj_speed']
    
    level_factor = (2 * data['pikachu_level'] / 5) + 2
    power = data['move_power'].fillna(0)
    data['core_base_damage'] = ((level_factor * power * data['attack_defense_ratio']) / 50) + 2

    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score']

y_train = train_feat[TARGET]
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)
test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Universal Categorical Encoding 
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# 5. Out-of-Fold 5-Fold Training
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
test_predictions = np.zeros(len(X_test))

print("Training Pure LightGBM Ensemble...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    
    # Highly Tuned LightGBM
    model = LGBMRegressor(
        n_estimators=600,        
        learning_rate=0.02,     
        max_depth=7,             
        num_leaves=45,           # Locked strictly into the generalization sweet spot
        min_child_samples=30,    # Forces leaves to have at least 30 samples 
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,           
        reg_lambda=2.0,          
        random_state=42 + fold,
        verbose=-1
    )
    
    model.fit(X_tr, y_tr)
    test_predictions += model.predict(X_test) / N_SPLITS

# 6. Generate Final Predictions
clipped_predictions = np.clip(test_predictions, 0, test_max_hp)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})
submission[TARGET] = submission[TARGET].fillna(0)
submission.to_csv('submissionDay14.csv', index=False)
print("Saved predictions to 'submissionDay14.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_27892\1409171786.py:90: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training Pure LightGBM Ensemble...
Saved predictions to 'submissionDay14.csv'


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor

# 1. Load Data
print("Loading data for the final Pseudo-Labeling run...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Champion Feature Engineering
def engineer_features(df):
    data = df.copy()
    data['damage_is_missing'] = data['damage_dealt'].isna().astype(int)
    data['healing_is_missing'] = data['healing_applied'].isna().astype(int)
    safe_damage = data['damage_dealt'].fillna(0)
    safe_healing = data['healing_applied'].fillna(0)
    safe_prev_hp = data['previous_hp'].fillna(data['max_hp'] if 'max_hp' in data.columns else 100)
    data['net_hp_change'] = safe_healing - safe_damage
    data['projected_hp'] = safe_prev_hp + data['net_hp_change']
    
    def calc_stage_multiplier(stage):
        return np.where(stage >= 0, (2 + stage) / 2.0, 2.0 / (2.0 - stage))
    
    data['true_attack'] = data['attack_stat'] * calc_stage_multiplier(data['attack_stage'].fillna(0))
    data['true_defense'] = data['defense_stat'] * calc_stage_multiplier(data['defense_stage'].fillna(0))
    data['true_speed'] = data['speed_stat_pikachu'] * calc_stage_multiplier(data['speed_stage'].fillna(0))
    data['adj_speed'] = np.where(data['pikachu_status'] == 'Paralyzed', data['true_speed'] * 0.5, data['true_speed'])
    data['adj_attack'] = np.where(data['pikachu_status'] == 'Burned', data['true_attack'] * 0.5, data['true_attack'])
    data['speed_advantage'] = data['adj_speed'] - data['speed_stat_opponent']
    data['level_ratio'] = data['pikachu_level'] / (data['opponent_level'] + 1e-5)
    data['attack_defense_ratio'] = np.where(
        data['move_category'] == 'Special',
        data['sp_attack_stat'] / (data['sp_defense_stat'] + 1e-5),
        data['adj_attack'] / (data['true_defense'] + 1e-5)
    )
    data['stab_multiplier'] = np.where(data['move_type'] == 'Electric', 1.5, 1.0)
    data['crit_multiplier'] = np.where(data['critical_hit'] == 1, 1.5, 1.0)
    data['battle_fatigue'] = data['round'].fillna(1) * data['turn'].fillna(1)
    data['exponential_fatigue'] = data['battle_fatigue'] ** 2
    
    danger = np.ones(len(data))
    danger = np.where((data['weather_condition'] == 'Rain') & (data['opponent_type'] == 'Water'), 1.5, danger)
    danger = np.where((data['weather_condition'] == 'Sun') & (data['opponent_type'] == 'Fire'), 1.5, danger)
    danger = np.where(data['weather_condition'].isin(['Hail', 'Sandstorm']), 1.2, danger)
    danger = np.where(data['terrain_type'] == 'Electric Terrain', 0.8, danger)
    data['weather_danger_level'] = danger
    
    data['attacks_first'] = np.where(data['speed_advantage'] > 0, 1, 0)
    data['weather_move_synergy'] = data['weather_condition'].astype(str) + "_" + data['move_type'].astype(str)
    data['type_matchup_str'] = data['move_type'].astype(str) + "_vs_" + data['opponent_type'].astype(str)
    data['raw_kinetic_energy'] = data['move_power'].fillna(0) * data['true_attack']
    data['log_attack_defense_ratio'] = np.log1p(data['attack_defense_ratio'])
    data['hp_deficit'] = data['max_hp'] - data['previous_hp'] if 'max_hp' in data.columns else 0
    data['level_scaled_attack'] = data['true_attack'] * data['pikachu_level']
    data['speed_ratio'] = data['adj_speed'] / (data['speed_stat_opponent'] + 1e-5)
    data['stat_product'] = data['true_attack'] * data['true_defense'] * data['adj_speed']
    
    level_factor = (2 * data['pikachu_level'] / 5) + 2
    power = data['move_power'].fillna(0)
    data['core_base_damage'] = ((level_factor * power * data['attack_defense_ratio']) / 50) + 2
    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

# 3. Setup Target
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAKED_FEATURES = ['trainer_focus_score']

y_train = train_feat[TARGET] 
X_train = train_feat.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)
test_ids = test_feat[ID_COL]
X_test = test_feat.drop(columns=[ID_COL] + LEAKED_FEATURES, errors='ignore')
test_max_hp = test_feat['max_hp'].fillna(test_feat['max_hp'].median())

# 4. Universal Categorical Encoding 
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# ---------------------------------------------------------
# PHASE 1: GENERATE PSEUDO-LABELS (Using a fast base model)
# ---------------------------------------------------------
print("Phase 1: Generating Pseudo-Labels from the Test Set...")
base_model = LGBMRegressor(
    n_estimators=600, learning_rate=0.02, max_depth=7, num_leaves=45, 
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8, 
    reg_alpha=0.1, reg_lambda=2.0, random_state=42, verbose=-1
)
base_model.fit(X_train, y_train)
pseudo_labels = base_model.predict(X_test)

# ---------------------------------------------------------
# PHASE 2: COMBINE DATASETS
# ---------------------------------------------------------
print("Phase 2: Constructing the Massive Combined Dataset...")
pseudo_y_train = pd.Series(pseudo_labels, name=TARGET)

X_train_massive = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
y_train_massive = pd.concat([y_train, pseudo_y_train], axis=0).reset_index(drop=True)

# THE FIX: Re-cast the destroyed categorical columns back to 'category'
for col in categorical_cols:
    X_train_massive[col] = X_train_massive[col].astype('category')

# ---------------------------------------------------------
# PHASE 3: TRAIN THE FINAL SUPER-MODEL
# ---------------------------------------------------------
print("Phase 3: Training Final 3-Seed Super-Model...")
SEEDS = [42, 1337, 2026]
final_test_predictions = np.zeros(len(X_test))

N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=99)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_massive)):
    X_tr, y_tr = X_train_massive.iloc[train_idx], y_train_massive.iloc[train_idx]
    
    fold_preds = np.zeros(len(X_test))
    
    for seed in SEEDS:
        model = LGBMRegressor(
            n_estimators=600, learning_rate=0.02, max_depth=7, num_leaves=45, 
            min_child_samples=30, subsample=0.8, colsample_bytree=0.8, 
            reg_alpha=0.1, reg_lambda=2.0, random_state=seed, verbose=-1
        )
        model.fit(X_tr, y_tr)
        fold_preds += model.predict(X_test) / len(SEEDS)
        
    final_test_predictions += fold_preds / N_SPLITS

# 5. Apply Bounds & Export
clipped_predictions = np.clip(final_test_predictions, 0, test_max_hp)
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: clipped_predictions
})
submission[TARGET] = submission[TARGET].fillna(0)
submission.to_csv('submissionDay15.csv', index=False)
print("Saved predictions to 'submission_pseudo_label_endgame.csv'")

Loading data for the final Pseudo-Labeling run...


C:\Users\Bhavin\AppData\Local\Temp\ipykernel_316\4120425279.py:79: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Phase 1: Generating Pseudo-Labels from the Test Set...
Phase 2: Constructing the Massive Combined Dataset...
Phase 3: Training Final 3-Seed Super-Model...
Saved predictions to 'submission_pseudo_label_endgame.csv'


In [ ]:
import pandas as pd
import numpy as np
import optuna

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor


# =========================================================
# 1. LOAD DATA
# =========================================================

print("Loading data...")

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")


# =========================================================
# 2. FEATURE ENGINEERING
# =========================================================

def engineer_features(df):

    data = df.copy()

    data["damage_is_missing"] = data["damage_dealt"].isna().astype(int)
    data["healing_is_missing"] = data["healing_applied"].isna().astype(int)

    safe_damage = data["damage_dealt"].fillna(0)
    safe_healing = data["healing_applied"].fillna(0)

    safe_prev_hp = data["previous_hp"].fillna(
        data["max_hp"] if "max_hp" in data.columns else 100
    )

    data["net_hp_change"] = safe_healing - safe_damage
    data["projected_hp"] = safe_prev_hp + data["net_hp_change"]

    def calc_stage_multiplier(stage):
        return np.where(
            stage >= 0,
            (2 + stage) / 2.0,
            2.0 / (2.0 - stage)
        )

    data["true_attack"] = (
        data["attack_stat"] *
        calc_stage_multiplier(data["attack_stage"].fillna(0))
    )

    data["true_defense"] = (
        data["defense_stat"] *
        calc_stage_multiplier(data["defense_stage"].fillna(0))
    )

    data["true_speed"] = (
        data["speed_stat_pikachu"] *
        calc_stage_multiplier(data["speed_stage"].fillna(0))
    )

    data["adj_speed"] = np.where(
        data["pikachu_status"] == "Paralyzed",
        data["true_speed"] * 0.5,
        data["true_speed"]
    )

    data["adj_attack"] = np.where(
        data["pikachu_status"] == "Burned",
        data["true_attack"] * 0.5,
        data["true_attack"]
    )

    data["speed_advantage"] = (
        data["adj_speed"] -
        data["speed_stat_opponent"]
    )

    data["level_ratio"] = (
        data["pikachu_level"] /
        (data["opponent_level"] + 1e-5)
    )

    data["attack_defense_ratio"] = np.where(
        data["move_category"] == "Special",
        data["sp_attack_stat"] /
        (data["sp_defense_stat"] + 1e-5),

        data["adj_attack"] /
        (data["true_defense"] + 1e-5)
    )

    data["stab_multiplier"] = np.where(
        data["move_type"] == "Electric",
        1.5,
        1.0
    )

    data["crit_multiplier"] = np.where(
        data["critical_hit"] == 1,
        1.5,
        1.0
    )

    data["battle_fatigue"] = (
        data["round"].fillna(1) *
        data["turn"].fillna(1)
    )

    data["exponential_fatigue"] = (
        data["battle_fatigue"] ** 2
    )

    danger = np.ones(len(data))

    danger = np.where(
        (data["weather_condition"] == "Rain") &
        (data["opponent_type"] == "Water"),
        1.5,
        danger
    )

    danger = np.where(
        (data["weather_condition"] == "Sun") &
        (data["opponent_type"] == "Fire"),
        1.5,
        danger
    )

    danger = np.where(
        data["weather_condition"].isin(["Hail", "Sandstorm"]),
        1.2,
        danger
    )

    danger = np.where(
        data["terrain_type"] == "Electric Terrain",
        0.8,
        danger
    )

    data["weather_danger_level"] = danger

    data["attacks_first"] = np.where(
        data["speed_advantage"] > 0,
        1,
        0
    )

    data["weather_move_synergy"] = (
        data["weather_condition"].astype(str)
        + "_"
        + data["move_type"].astype(str)
    )

    data["type_matchup_str"] = (
        data["move_type"].astype(str)
        + "_vs_"
        + data["opponent_type"].astype(str)
    )

    data["raw_kinetic_energy"] = (
        data["move_power"].fillna(0) *
        data["true_attack"]
    )

    data["log_attack_defense_ratio"] = np.log1p(
        data["attack_defense_ratio"].clip(lower=0)
    )

    data["hp_deficit"] = (
        data["max_hp"] -
        data["previous_hp"]
        if "max_hp" in data.columns
        else 0
    )

    data["level_scaled_attack"] = (
        data["true_attack"] *
        data["pikachu_level"]
    )

    data["speed_ratio"] = (
        data["adj_speed"] /
        (data["speed_stat_opponent"] + 1e-5)
    )

    data["stat_product"] = (
        data["true_attack"] *
        data["true_defense"] *
        data["adj_speed"]
    )

    level_factor = (
        2 * data["pikachu_level"] / 5
    ) + 2

    power = data["move_power"].fillna(0)

    data["core_base_damage"] = (
        (level_factor *
         power *
         data["attack_defense_ratio"]) / 50
    ) + 2

    return data


train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)


# =========================================================
# 3. DATA PREPARATION
# =========================================================

TARGET = "pikachu_hp"
ID_COL = "battle_turn"

LEAKED_FEATURES = ["trainer_focus_score"]

y = train_feat[TARGET].copy()

X = train_feat.drop(
    columns=[TARGET, ID_COL] + LEAKED_FEATURES,
    errors="ignore"
)

X_test = test_feat.drop(
    columns=[ID_COL] + LEAKED_FEATURES,
    errors="ignore"
)

test_ids = test_feat[ID_COL].copy()

test_max_hp = test_feat["max_hp"].fillna(
    test_feat["max_hp"].median()
)


# Make categorical columns consistent
categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in categorical_cols:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")


# =========================================================
# 4. OPTUNA
#    Tune around your CURRENT champion
# =========================================================

print("\nStarting Optuna tuning...")

N_TUNING_FOLDS = 5

kf_tune = KFold(
    n_splits=N_TUNING_FOLDS,
    shuffle=True,
    random_state=99
)


def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 400, 1000
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.008,
            0.04,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            5,
            10
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            25,
            80
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            15,
            70
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.70,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.70,
            1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0.001,
            1.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.5,
            8.0,
            log=True
        ),

        "random_state": 42,
        "verbosity": -1
    }

    scores = []

    for train_idx, val_idx in kf_tune.split(X):

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_tr = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model = LGBMRegressor(**params)

        model.fit(X_tr, y_tr)

        pred = model.predict(X_val)

        scores.append(
            r2_score(y_val, pred)
        )

    return np.mean(scores)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True
)

BEST_PARAMS = study.best_params

print("\n====================================")
print("BEST CV SCORE:", study.best_value)
print("BEST PARAMETERS:")
for k, v in BEST_PARAMS.items():
    print(f"{k}: {v}")
print("====================================")


# =========================================================
# 5. GENERATE BETTER PSEUDO-LABELS
#    5 folds × 3 seeds
# =========================================================

print("\nGenerating ensemble pseudo-labels...")

PL_SEEDS = [42, 1337, 2026]

test_predictions = []

for seed in PL_SEEDS:

    fold_test_preds = np.zeros(len(X_test))

    for fold, (train_idx, val_idx) in enumerate(
        kf_tune.split(X)
    ):

        model_params = BEST_PARAMS.copy()
        model_params["random_state"] = seed
        model_params["verbosity"] = -1

        model = LGBMRegressor(
            **model_params
        )

        model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx]
        )

        fold_test_preds += (
            model.predict(X_test) /
            N_TUNING_FOLDS
        )

    test_predictions.append(
        fold_test_preds
    )


test_predictions = np.array(test_predictions)

pseudo_labels = test_predictions.mean(axis=0)

# Model disagreement = approximate uncertainty
pseudo_uncertainty = test_predictions.std(axis=0)


# =========================================================
# 6. CONFIDENCE FILTER
# =========================================================

# Keep the most stable 70% of pseudo-labels
threshold = np.percentile(
    pseudo_uncertainty,
    70
)

confidence_mask = (
    pseudo_uncertainty <= threshold
)

print(
    f"\nUsing {confidence_mask.mean() * 100:.1f}% "
    "of test rows as pseudo-labels."
)


X_test_confident = X_test.loc[
    confidence_mask
].copy()

pseudo_y_confident = pd.Series(
    pseudo_labels[confidence_mask],
    name=TARGET
)


# =========================================================
# 7. COMBINE REAL + PSEUDO DATA
# =========================================================

X_massive = pd.concat(
    [
        X.reset_index(drop=True),
        X_test_confident.reset_index(drop=True)
    ],
    axis=0
)

y_massive = pd.concat(
    [
        y.reset_index(drop=True),
        pseudo_y_confident.reset_index(drop=True)
    ],
    axis=0
)

# Real labels = 1.0
# Pseudo labels = 0.20
sample_weights = np.concatenate(
    [
        np.ones(len(X)),
        np.full(
            len(X_test_confident),
            0.20
        )
    ]
)

# Re-cast categories after concat
for col in categorical_cols:
    X_massive[col] = X_massive[col].astype("category")


# =========================================================
# 8. FINAL 5-FOLD × 3-SEED MODEL
# =========================================================

print("\nTraining final model...")

SEEDS = [42, 1337, 2026]

N_FINAL_FOLDS = 5

kf_final = KFold(
    n_splits=N_FINAL_FOLDS,
    shuffle=True,
    random_state=99
)

final_predictions = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(
    kf_final.split(X_massive)
):

    print(
        f"Fold {fold + 1}/{N_FINAL_FOLDS}"
    )

    X_tr = X_massive.iloc[train_idx]
    y_tr = y_massive.iloc[train_idx]

    weights_tr = sample_weights[train_idx]

    fold_prediction = np.zeros(len(X_test))

    for seed in SEEDS:

        model_params = BEST_PARAMS.copy()
        model_params["random_state"] = seed
        model_params["verbosity"] = -1

        model = LGBMRegressor(
            **model_params
        )

        model.fit(
            X_tr,
            y_tr,
            sample_weight=weights_tr
        )

        fold_prediction += (
            model.predict(X_test) /
            len(SEEDS)
        )

    final_predictions += (
        fold_prediction /
        N_FINAL_FOLDS
    )


# =========================================================
# 9. BOUNDS + SUBMISSION
# =========================================================

final_predictions = np.clip(
    final_predictions,
    0,
    test_max_hp.values
)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: final_predictions
})

submission[TARGET] = submission[TARGET].fillna(0)

submission.to_csv(
    "submissionDay16.csv",
    index=False
)

print(
    "\nSaved: submission_optuna_pseudo.csv"
)

1. Loading data...
2. Initializing Level 1 Models...
3. Generating Out-of-Fold Meta-Features...
   -> Training lgbm_deep...
   -> Training lgbm_shallow...
   -> Training catboost...
4. Training Level 2 Meta-Learner (Bayesian Ridge)...
Meta-Model Weights Assigned to [LGBM_Deep, LGBM_Shallow, CatBoost]: [ 0.75870229 -0.15758424  0.40311327]
SUCCESS: Saved predictions to 'submission_oof_stacking.csv'
